# An introduction to 3D LUTs with OpenColorIO

This notebook shows how to

- construct a 3D LUT with an 'identity transform', that is, a 3D LUT that, given an input, produces as a result that same input. This sounds useless, but it's actually very useful because you can modify the transform, either all the cells in the 3D lattice at once, or selectively.
- modify the contents of a 3D LUT, as mentioned above. The example here is to produce an image where the green channel is flipped. The algorithm is basically:
  - take an RGB to RGB identity LUT where the domain (the input) has values from 0.0 to 1.0 inclusive, and the range (the output) has values from 0.0 to 1.0 inclusive.
  - for all the cells in the 3D lattice, change whatever green value 'g' is there, to 1.0 - g.
  - return the LUT, now that it is capable of inverting the green channel
- write out a 3D LUT defined as above to a file, so other people can use it without having to know how you made it
- read in the 3D LUT, read in a 16-bit TIFF image file, apply the 3D LUT to the image, and write out the result as a new 16-bit image file

As a check, the notebook takes an existing image file (Utsi holding a Macbeth chart near a dam), inverts its green channel and writes that image out; then reads in the image with the inverted green channel, applies the LUT again (which is an inversion of an inversion, so the result is to undo the change from the image completely) and then writes out the result. The original image, and the image with the inverted inverted green channel, will match.

In [1]:
from pathlib import Path

import numpy as np
import OpenImageIO as oiio
import PyOpenColorIO as ocio

CUBE_EDGE_LENGTH = 33

G_INVERTING_CUBE_PATH = Path('/tmp/g_inverting_cube.ctf')
SAMPLE_IN_IMAGE_PATH = Path('/tmp/utsi_art_generated_709_quarter_res.tif')
SAMPLE_OUT_IMAGE_PATH = Path('/tmp/utsi_g_flipped_art_generated_709_quarter_res.tif')
SAMPLE_ROUND_TRIP_IMAGE_PATH = Path('/tmp/utsi_rt_flipped_art_generated_709_quarter_res.tif')


In [2]:
def make_g_inverting_cube_transform(n: int) -> ocio.Lut3DTransform:
    cube = ocio.Lut3DTransform(gridSize=n)
    cube_data = cube.getData().reshape(n, n, n, 3)
    cube_data[:,:,:,1] = 1.0 - cube_data[:,:,:,1]  # invert the G channel (as in, g = 1.0 - g)
    cube.setData(cube_data)
    return cube


In [3]:
def write_g_inverting_cube(path: Path) -> None:
    # Only group transforms can be written, so wrap 3D LUT transform as the only member of a group
    gt = ocio.GroupTransform()
    lut3d_transform = make_g_inverting_cube_transform(CUBE_EDGE_LENGTH)
    gt.appendTransform(lut3d_transform)
    gt.write(formatName="Color Transform Format",
             config=ocio.Config.CreateRaw(),
             fileName=str(path)
    )
    

In [4]:
def apply_g_inverting_cube_to_image(in_path: Path, out_path: Path) -> None:
    config = ocio.Config().CreateRaw()
    fileTransform = ocio.FileTransform(str(G_INVERTING_CUBE_PATH))
    proc = config.getProcessor(fileTransform)
    cpu = proc.getDefaultCPUProcessor()
    buf = oiio.ImageBuf(str(in_path))
    buf.make_writable()
    pixels = buf.get_pixels(oiio.FLOAT)
    # You may see warnings from VS Code's Pylance (Python syntax / semantics checking
    # module) about the next two lines. Ignore them. I am looking into providing Pylance
    # with enough corrected information so that the spurious warnings go away.
    cpu.applyRGB(pixels)
    out_buf = oiio.ImageBuf(pixels)
    out_buf.write(str(out_path), dtype=oiio.UINT16)


### Create the green-channel-inverting 3D LUT, and write it out

In [5]:
write_g_inverting_cube(G_INVERTING_CUBE_PATH)

### Apply the 3D LUT to the original image

This will produce a new image, with the green channel flipped

In [6]:
apply_g_inverting_cube_to_image(in_path=SAMPLE_IN_IMAGE_PATH,
                                out_path=SAMPLE_OUT_IMAGE_PATH)

### Apply the 3D LUT again, to the image produced above

This will produce a third image, matching the original.

In [7]:
apply_g_inverting_cube_to_image(in_path=SAMPLE_OUT_IMAGE_PATH,
                                out_path=SAMPLE_ROUND_TRIP_IMAGE_PATH)